In [2]:
import os
import json
from litellm import headers
from urllib.parse import quote
import requests
from requests.structures import CaseInsensitiveDict
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import sqlite3
import base64
from huggingface_hub import InferenceClient

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
places_api_key = os.getenv("PLACES_API_KEY")
huggingface_api_key = os.getenv("HUGGINGFACE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if client_id and client_secret:
    print(f"XWeather API Key exists and begins {client_id[:8]} and {client_secret[:8]}")
else:
    print("XWeather API Key not set")

if places_api_key:
    print(f"Places API Key exists and begins {places_api_key[:8]}")
else:
    print("Places API Key not set")

if huggingface_api_key:
    print(f"Huggingface API Key exists and begins {places_api_key[:8]}")
else:
    print("Huggingface API Key not set")

if openrouter_api_key:
    print(f"Openrouter API Key exists and begins {places_api_key[:8]}")
else:
    print("Openrouter API Key not set")
    

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"

MODEL = "gemini-3.1-flash-lite"
VOICE_MODEL = "deepgram/flux-tts:free"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

Google API Key exists and begins AQ.Ab8RN
XWeather API Key exists and begins vuQ0B1tv and YR1tpsnl
Places API Key exists and begins 6955b278
Huggingface API Key exists and begins 6955b278
Openrouter API Key exists and begins 6955b278


In [3]:
system_message = """
You are a helpful assistant that helps users plan their vacations called VacationAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def get_weather(location):

    weather_response = requests.get(f'https://data.api.xweather.com/observations/{location}?client_id={client_id}&client_secret={client_secret}')
    print("Weather API Response:", weather_response.json()) 

    if weather_response.status_code == 200:
        data = weather_response.json()
        if data['success']:
            ob = data['response']['ob']
            return f"The current weather in {location} is {ob['weather'].lower()} with a temperature of {ob['tempF']}"
        else:
            return f"An error occurred: {data['error']['description']}"
    else:
        return f"An error occurred while fetching data: HTTP status code %d" % weather_response.status_code


In [5]:
def get_travel_locations_for_destination(destination):

    headers = {"Accept": "application/json"}

    encoded_dest = quote(destination)
    geocode_url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_dest}&limit=1&format=json&apiKey={places_api_key}"
    geo_response = requests.get(geocode_url, headers=headers)

    if geo_response.status_code != 200:
        return f"Geocoding error: HTTP status code {geo_response.status_code}"
        
    geo_data = geo_response.json()
    if not geo_data.get("results"):
        return f"Could not find coordinates for destination: {destination}"

    place_id = geo_data["results"][0]["place_id"]
    places_url = f"https://api.geoapify.com/v2/places?categories=tourism.attraction&filter=place:{place_id}&limit=20&apiKey={places_api_key}"
    
    response = requests.get(places_url, headers=headers)

    if response.status_code == 200:
        return response.json()
    else:
        return f"An error occurred while fetching travel locations: HTTP status code {response.status_code}"


In [6]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


In [7]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [8]:
weather_function = {
    "name": "get_weather",
    "description": "Get the current weather for a given location.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The location for which to get weather information",
            },
        },
        "required": ["location"],
        "additionalProperties": False
    }
}

travel_locations_function = {
    "name": "get_travel_locations",
    "description": "Get a list of travel locations (tourism attractions) for a given destination.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination"],
        "additionalProperties": False
    }
}

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

update_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The price of a return ticket to the destination city",
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": weather_function}, {"type": "function", "function": travel_locations_function},{"type": "function", "function": price_function}, {"type": "function", "function": update_price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_weather',
   'description': 'Get the current weather for a given location.',
   'parameters': {'type': 'object',
    'properties': {'location': {'type': 'string',
      'description': 'The location for which to get weather information'}},
    'required': ['location'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_travel_locations',
   'description': 'Get a list of travel locations (tourism attractions) for a given destination.',
   'parameters': {'type': 'object',
    'properties': {'destination': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
     

In [9]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_weather":
            arguments = json.loads(tool_call.function.arguments)
            location = arguments.get('location')
            cities.append(location)
            weather_details = get_weather(location)
            responses.append({
                "role": "tool",
                "content": weather_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_travel_locations":
            arguments = json.loads(tool_call.function.arguments)
            destination = arguments.get('destination')
            cities.append(destination)
            travel_locations = get_travel_locations_for_destination(destination)
            responses.append({
                "role": "tool",
                "content": json.dumps(travel_locations),
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            cities.append(city)
            set_ticket_price(city, price)
            price_details = f"Ticket price for {city} has been set to ${price}"
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [10]:
def draw_image(city):
    client = InferenceClient(provider="auto",api_key=huggingface_api_key)
    prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"
    print("Sending request to Hugging Face serverless API...")

    try:
        # Request the image from the API (returns a PIL Image directly)
        image = client.text_to_image(prompt, model="black-forest-labs/FLUX.1-schnell")
        return image

    except Exception as e:
        print(f"An error occurred: {e}")

In [11]:
def voice_assistant(message):
    response = openrouter.audio.speech.create(
      model=VOICE_MODEL,
      voice="flux-alexis-en",
      input=message
    )
    return response.content

In [12]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = voice_assistant(reply)

    if cities:
        image = draw_image(cities[0])

    return history, voice, image

In [17]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks(title="VacationAI") as ui:
    gr.Markdown("VacationAI: Your goto Vacation Planning Assistant")
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))



* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Weather API Response: {'success': True, 'error': None, 'response': {'id': 'EDDB', 'dataSource': 'METAR_NOAA', 'loc': {'long': 13.533333333333, 'lat': 52.383333333333}, 'place': {'name': 'berlin/schonefel', 'city': 'berlin/schonefel', 'state': '', 'country': 'de'}, 'profile': {'tz': 'Europe/Berlin', 'tzname': 'CEST', 'tzoffset': 7200, 'isDST': True, 'elevM': 48, 'elevFT': 157}, 'obTimestamp': 1789321800, 'obDateTime': '2026-09-13T19:50:00+02:00', 'ob': {'type': 'station', 'timestamp': 1789321800, 'dateTimeISO': '2026-09-13T19:50:00+02:00', 'recTimestamp': 1789322108, 'recDateTimeISO': '2026-09-13T19:55:08+02:00', 'tempC': 19, 'tempF': 66, 'dewpointC': 15, 'dewpointF': 59, 'humidity': 78, 'pressureMB': 1017, 'pressureIN': 30.03, 'spressureMB': 1011, 'spressureIN': 29.86, 'altimeterMB': 1017, 'altimeterIN': 30.03, 'windKTS': 8, 'windKPH': 15, 'windMPH': 9, 'windMPS': 4.12, 'windSpeedKTS': 8, 'windSpeedKPH': 15, 'windSpeedMPH': 9, 'windSpeedMPS': 4.12, 'windDirDEG': 230, 'windDir': 'SW', '